## 연습 1: 접근 제어

당신은 **NorthMart Retail**의 데이터 엔지니어입니다. NorthMart Retail은 전국 슈퍼마켓 체인입니다. 팀은 Unity Catalog에 고객 및 거래 데이터를 저장합니다. 분석가가 쿼리를 실행할 수 있으려면 먼저 카탈로그, 스키마를 생성하고 고객 테이블을 채운 다음 `retail-analysts` 그룹에 필요한 권한을 부여해야 합니다.

> ⚠️ 이 연습을 시작하기 전에 Databricks에서 `retail-analysts` 그룹을 생성했는지 확인하고 자신의 사용자 계정을 추가했습니다. 랩 설정 지침을 참고하세요.

### 작업 1.1 — 카탈로그, 스키마 및 고객 테이블 생성

아래 셀은 이 연습에 필요한 모든 객체를 설정합니다:
- `retail_catalog`라는 카탈로그를 생성하여 모든 NorthMart 데이터 자산을 보유합니다.
- 내부에 `security_lab`이라는 스키마를 생성합니다.
- 관리형 Delta 테이블 `retail_catalog.security_lab.customers`를 생성하고 6개의 샘플 고객 기록으로 채웁니다. 이러한 행은 이 연습의 나머지 부분에서 행 필터링 및 열 마스킹을 시연하는 데 사용됩니다.

| customer_id | customer_name   | email                   | region | total_spend | tier     |
|-------------|-----------------|-------------------------|--------|-------------|----------|
| 1           | Alice Johnson   | alice@northmart.com     | North  | 5200.00     | Standard |
| 2           | Bob Martinez    | bob@northmart.com       | South  | 8100.00     | Premium  |
| 3           | Carol Lee       | carol@northmart.com     | North  | 3400.00     | Standard |
| 4           | David Chen      | david@northmart.com     | East   | 7800.00     | Premium  |
| 5           | Eve Patel       | eve@northmart.com       | West   | 6200.00     | Standard |
| 6           | Frank Kim       | frank@northmart.com     | South  | 4100.00     | Standard |

셀을 실행하여 이러한 객체를 프로비저닝합니다.

In [ ]:
# 기본 저장소가 활성화되지 않았으므로 기본 카탈로그의 루트에서 저장소 경로를 상속합니다.
catalogs = [catalog.name for catalog in spark.catalog.listCatalogs("adb_dp750*")]
dp750_default_catalog = catalogs[0] if catalogs else None

storage_root = (
    spark.sql(f"DESCRIBE CATALOG EXTENDED {dp750_default_catalog}")
    .filter("info_name = 'Storage Root'")
    .select("info_value")
    .first()[0]
)
print (f"저장소 루트: {storage_root}")

spark.sql(f"""
    CREATE CATALOG IF NOT EXISTS retail_catalog
    MANAGED LOCATION '{storage_root}'
    COMMENT 'NorthMart Retail 데이터 자산용 카탈로그'
""")

In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS retail_catalog.security_lab
  COMMENT '보안 랩 연습에 사용되는 스키마';

CREATE OR REPLACE TABLE retail_catalog.security_lab.customers AS
SELECT * FROM VALUES
  (1, 'Alice Johnson', 'alice@northmart.com', 'North', 5200.00, 'Standard'),
  (2, 'Bob Martinez',  'bob@northmart.com',   'South', 8100.00, 'Premium'),
  (3, 'Carol Lee',     'carol@northmart.com', 'North', 3400.00, 'Standard'),
  (4, 'David Chen',    'david@northmart.com', 'East',  7800.00, 'Premium'),
  (5, 'Eve Patel',     'eve@northmart.com',   'West',  6200.00, 'Standard'),
  (6, 'Frank Kim',     'frank@northmart.com', 'South', 4100.00, 'Standard')
  AS t(customer_id, customer_name, email, region, total_spend, tier);

### 작업 1.2 — retail-analysts 그룹에 권한 부여

`retail-analysts` 그룹은 `customers` 테이블을 쿼리해야 합니다. Unity Catalog 계층 구조의 모든 필수 수준에서 필요한 권한을 부여합니다:

- `retail_catalog`에 대한 `USE CATALOG`
- `retail_catalog.security_lab`에 대한 `USE SCHEMA`
- `retail_catalog.security_lab.customers`에 대한 `SELECT`

> 🤖 **Genie Code 팁:** 질문하세요:
> *"Databricks 그룹이 Unity Catalog 테이블에서 SELECT하도록 허용하는 데 필요한 SQL 권한은?"*

**힌트:** `GRANT <권한> ON <객체 유형> <객체 이름> TO \`retail-analysts\``을 사용하세요.

In [ ]:
%sql
-- TODO: 부여된 권한 확인

### 작업 1.3 — 부여된 권한 확인

권한이 올바르게 할당되었는지 확인하세요. `SHOW GRANTS`를 사용하여 `customers` 테이블의 모든 권한을 표시합니다.

> 🤖 **Genie Code 팁:** 질문하세요:
> *"SHOW GRANTS를 사용하여 Unity Catalog 테이블에 대한 권한을 확인하는 방법은?"*

**예상 결과:** `SELECT` 권한이 있는 `retail-analysts`가 나열되어야 합니다.

In [ ]:
%sql
-- TODO: retail_catalog.security_lab.customers의 모든 권한 표시

## 연습 2: 행 필터링

NorthMart Retail의 지역 분석가는 자신의 지역에 대한 고객 기록만 보아야 합니다. 쿼리하는 사람에 따라 행을 제한하는 **행 필터 함수**를 구현하겠습니다.

여러 사용자 계정을 생성할 수 없으므로 자신의 사용자 계정을 제외하고 모든 사람에게 `North` 지역 행만 표시하는 필터를 작성하고, 자신의 계정은 모든 행을 봅니다. 이것은 지역 분석가가 경험할 것을 시뮬레이션합니다.

### 작업 2.1 — 행 필터 함수 생성

SQL 함수 `retail_catalog.security_lab.region_filter`를 생성하여 `STRING` 유형의 `region` 매개변수를 사용하고 다음의 경우 `TRUE`를 반환합니다:
- `region`이 `'North'`이거나, **또는**
- 현재 사용자가 자신의 계정인 경우 (`your@email.com`을 실제 이메일로 바꾸세요).

이는 다음을 의미합니다: 권한이 있는 사용자가 아닌 한 `North` 지역 행만 표시됩니다.

> 🤖 **Genie Code 팁:** 질문하세요:
> *"Databricks Unity Catalog에서 current_user()를 사용하여 액세스를 제어하는 행 필터 함수를 생성하는 방법은?"*

**힌트:** `CREATE OR REPLACE FUNCTION ... RETURN <current_user()를 사용한 부울 식>`을 사용하세요.

In [ ]:
%sql
-- TODO: retail_catalog.security_lab에서 region_filter 함수를 생성하세요
-- 'your@email.com'을 실제 Databricks 사용자 이메일로 바꾸세요

### 작업 2.2 — 행 필터를 customers 테이블에 적용

해당 테이블에 대한 모든 쿼리에 자동으로 적용되도록 `region_filter` 함수를 `customers` 테이블에 연결하세요.

> 🤖 **Genie Code 팁:** 질문하세요:
> *"Azure Databricks에서 Unity Catalog 테이블에 행 필터를 설정하는 방법은?"*

**힌트:** `ALTER TABLE ... SET ROW FILTER <함수> ON (<열>)`을 사용하세요.

In [ ]:
%sql
-- TODO: region_filter를 retail_catalog.security_lab.customers의 region 열에 적용합니다